# 05: Development and freezing (gates G1 and G3)

Design decisions
- Tasks: CDC Diabetes (task B) and BRFSS 2015 heart disease (task A). Only training-portion rows (dev-train and dev-cal) are kept in memory; calibration splits 1 and 2 and the test split are not loaded. Their explained subsamples are chosen from row indices alone.
- Gate G1: a tuned unconstrained XGBoost (depth 4, 6 or 8; 200 or 400 trees; learning rate 0.05 or 0.1) is trained on dev-train and scored on dev-cal; a constrained model passes if it loses at most 0.01 AUROC against the best of these.
- Gate G3 runs the whole SCAX pipeline inside development. The model is trained on dev-train with tau from 5-fold out-of-fold scores; the prior is built from the candidate sets of 1,000 dev-train patients; dev-cal is split at random into a certification part (80%) and an evaluation part (20%), certificates come from the certification part, and releases are evaluated on 1,000 patients of the evaluation part. Utility target (plan RQ2): at least half of the evaluation patients receive a certified negative-outcome explanation at alpha_0 = 0.05 with M = 10 and beta = 0.30.
- Configurations are tried in the plan's G3 order, starting at the prior weight (certificate-guided inflation and the most general explanation, first in that order, are not implemented): regularised XGBoost (100 trees of depth 4, min_child_weight 50, gamma 2, max_bin 32) with lambda = 0.5, then lambda = 0.25, then coarser thresholds (max_bin 16), then a shallower model (depth 3). The first that passes G1 and meets the utility target is frozen; if none meets it, the G1-passing configuration with the highest development release rate is frozen and the shortfall is recorded. Development results for every evaluated configuration are stored on Drive so that a rerun resumes.
- Fixed targets: delta_1 = delta_2 = 0.025; alpha_0 = 0.05 (grid 0.02, 0.05, 0.10); beta = 0.30, so alpha_1 = 0.70 for enrichment claims (grid 0.2, 0.3, 0.5); M = 10.
- Policy certification is split by kind: negative-outcome releases and enrichment releases are certified separately at delta_2 / 2 each, with their own alpha_sel. Each alpha_sel is the smallest value on a fixed grid (negative: 0.05, 0.075, 0.10; enrichment: 0.70, 0.75, 0.80, 0.85) that the development selective error would pass at the expected final number of accepted patients.
- Explained subsamples: 5,000 test patients (plan); calibration split 2 sized from the development acceptance rates so that at least 2,000 accepted patients in all and 1,000 of each kind are expected, at most 12,000.
- Final model: the frozen configuration trained on the whole training portion, tau from 5-fold out-of-fold scores, exported and checked against the native library on every training row and by the primary and second soundness checks on 300 boxes (Z3 on 30). The prior is built from the candidate sets of 2,000 training patients and every candidate box is re-checked by the second exact check.
- The frozen configuration of each task is written to configs/frozen/ and its SHA-256 to results/05_frozen.json; the prior file and the subsample index files are hashed into the configuration.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time, math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scax import paths
from scax.utils import seed_everything, sha256_array, sha256_file, write_json
from scax.data import public, splits
from scax.models import train
from scax.models.predictor import native_scores
from scax.experiments import development as dev
from scax.explain.candidates import CandidateGenerator
from scax.explain.verify import verify_boxes, drop_unverified, z3_sample
from scax.encode.interval_bound import IntervalBound
from scax.encode.cell_search import CellSearch
from scax.encode.real_search import RealSearch
from scax.encode.real_encoding import Z3RealEncoding
from scax.certify.clopper_pearson import cp_upper
from scax.frozen import write_frozen

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
M, DELTA1, DELTA2 = 10, 0.025, 0.025
ALPHA0_MAIN, ALPHA0_GRID = 0.05, [0.02, 0.05, 0.10]
BETA_MAIN, BETA_GRID = 0.30, [0.2, 0.3, 0.5]
LAMBDAS, M_LIST = [0.5, 0.25], [1, 5, 10]
TARGET_NEGATIVE_RELEASE, G1_MAX_LOSS = 0.50, 0.01
N_PRIOR_DEV, N_EVAL_DEV, N_PRIOR_FINAL, N_TEST_EXPLAIN, N2_CAP = 1000, 1000, 2000, 5000, 12000
ALPHA_SEL_GRID = {0: [0.05, 0.075, 0.10], 1: [0.70, 0.75, 0.80, 0.85]}
DEV_DIR = paths.DERIVED / 'development'; DEV_DIR.mkdir(parents=True, exist_ok=True)
FROZEN_DIR = paths.DERIVED / 'frozen'; FROZEN_DIR.mkdir(parents=True, exist_ok=True)
REC06S = json.load(open(f'{paths.RESULTS}/06_splits.json'))
CARDS = json.load(open(f'{paths.RESULTS}/01_dataset_cards.json'))
assert json.load(open(f'{paths.RESULTS}/06_encoding_tests.json'))['gate_G2']['passed']

## Training portion only, and the development partition

In [ ]:
DATA, PART = {}, {}
for task in TASKS:
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == REC06S['splits'][task]['sha256'], 'split differs from notebook 06'
    train_rows = splits.indices(code, 'train')
    Xt, yt, names = X.to_numpy(dtype=float)[train_rows], y[train_rows], list(X.columns)
    del X, y                                    # rows outside the training portion are not kept
    pos = lambda idx: np.searchsorted(train_rows, idx)
    dtr, dcal = pos(splits.indices(code, 'dev_train')), pos(splits.indices(code, 'dev_cal'))
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(dcal)
    n_cert = int(round(0.8 * len(dcal)))
    cert, pool = np.sort(perm[:n_cert]), np.sort(perm[n_cert:])
    ev = np.sort(rng.choice(pool, N_EVAL_DEV, replace=False))
    pr = np.sort(rng.choice(dtr, N_PRIOR_DEV, replace=False))
    DATA[task] = (Xt, yt, names, code, train_rows)
    PART[task] = {'dev_train': dtr, 'dev_cal': dcal, 'cert': cert, 'eval': ev, 'prior': pr}
    print(task, {k: len(v) for k, v in PART[task].items()}, 'training rows', len(train_rows), 'positives', int(yt.sum()))

## Gate G1: tuned unconstrained reference

In [ ]:
G1_REF, G1_TABLES = {}, []
for task in TASKS:
    Xt, yt, names, code, _ = DATA[task]; P = PART[task]
    t0 = time.time()
    ref = dev.tuned_reference(Xt[P['dev_train']], yt[P['dev_train']], Xt[P['dev_cal']], yt[P['dev_cal']], SEED)
    ref.insert(0, 'task', task)
    G1_TABLES.append(ref)
    G1_REF[task] = float(ref.auroc.max())
    print(f"{task}: best tuned AUROC {G1_REF[task]:.4f} ({time.time() - t0:.0f}s)")
    print(ref.head(3).round(4).to_string(index=False))
pd.concat(G1_TABLES).to_csv(f'{paths.RESULTS}/05_g1_reference.csv', index=False)

## Gate G3: the SCAX pipeline inside development, in the plan order

In [ ]:
DEV_TABLES, DEV_META, CHOICE = [], [], {}


def main_rows(tab):
    return tab[(tab.M == M) & (tab.alpha_0 == ALPHA0_MAIN) & (tab.beta == BETA_MAIN)]


for task in TASKS:
    Xt, yt, names, code, _ = DATA[task]; P = PART[task]
    evaluated = []
    for name, params in dev.CASCADE:
        f_tab, f_meta = DEV_DIR / f'{task}__{name}.csv', DEV_DIR / f'{task}__{name}.json'
        if f_tab.exists() and f_meta.exists():
            tab, meta = pd.read_csv(f_tab), json.load(open(f_meta))
            print(f'{task} {name}: loaded from Drive')
        else:
            quick = train.fit('xgboost', Xt[P['dev_train']], yt[P['dev_train']], params, SEED)
            auroc = roc_auc_score(yt[P['dev_cal']], native_scores(quick, 'xgboost', Xt[P['dev_cal']]))
            g1 = bool(auroc >= G1_REF[task] - G1_MAX_LOSS)
            if not g1:
                tab, meta = pd.DataFrame(), {'config': name, 'params': params, 'auroc_dev_cal': auroc}
            else:
                tab, meta = dev.evaluate_config(name, params, Xt, yt, P['dev_train'], P['cert'], P['eval'], P['prior'],
                                                names, SEED, M, DELTA1, DELTA2 / 2, ALPHA0_GRID, BETA_GRID,
                                                [1.0] + LAMBDAS, M_LIST)
            meta.update({'task': task, 'g1_reference': G1_REF[task], 'g1_pass': g1})
            tab.to_csv(f_tab, index=False)
            json.dump(meta, open(f_meta, 'w'), indent=1, default=float)
        meta['task'] = task
        DEV_META.append(meta)
        if not meta['g1_pass']:
            print(f"{task} {name}: fails G1 (AUROC {meta['auroc_dev_cal']:.4f} against {G1_REF[task]:.4f})"); continue
        tab.insert(0, 'task', task) if 'task' not in tab.columns else None
        DEV_TABLES.append(tab)
        mr = main_rows(tab)
        for lam in LAMBDAS:
            rate = float(mr[mr['lambda'] == lam].release_rate_class0.iloc[0])
            evaluated.append((rate, name, params, lam))
            print(f"{task} {name} lambda={lam}: negative-outcome release rate {rate:.3f}")
            if rate >= TARGET_NEGATIVE_RELEASE and task not in CHOICE:
                CHOICE[task] = {'config': name, 'params': params, 'lambda': lam, 'target_met': True}
        if task in CHOICE:
            break
    if task not in CHOICE:
        rate, name, params, lam = max(evaluated, key=lambda t: t[0])
        CHOICE[task] = {'config': name, 'params': params, 'lambda': lam, 'target_met': False}
    print(task, 'frozen choice:', CHOICE[task])
DEV_ALL = pd.concat(DEV_TABLES, ignore_index=True)
DEV_ALL.to_csv(f'{paths.RESULTS}/05_dev_results.csv', index=False)
pd.DataFrame(DEV_META).to_csv(f'{paths.RESULTS}/05_dev_configs.csv', index=False)

In [ ]:
show = DEV_ALL[(DEV_ALL.M == M) & (DEV_ALL.beta == BETA_MAIN)]
print(show[['task', 'config', 'lambda', 'alpha_0', 'release_rate', 'release_rate_class0', 'release_rate_class1',
            'selective_error_class0', 'selective_upper_class0', 'selective_error_class1', 'selective_upper_class1',
            'released_size_class0', 'released_support_class0']].round(3).to_string(index=False))
print(pd.DataFrame(DEV_META)[['task', 'config', 'g1_pass', 'auroc_dev_cal', 'g1_reference', 'thresholds',
                              'median_seconds_per_patient', 'distinct_boxes']].round(4).to_string(index=False))

## Policy targets and the size of the explained subsample of calibration split 2

In [ ]:
TARGETS = {}
for task in TASKS:
    ch = CHOICE[task]
    r = DEV_ALL[(DEV_ALL.task == task) & (DEV_ALL.config == ch['config']) & (DEV_ALL['lambda'] == ch['lambda'])
                & (DEV_ALL.M == M) & (DEV_ALL.alpha_0 == ALPHA0_MAIN) & (DEV_ALL.beta == BETA_MAIN)].iloc[0]
    code = DATA[task][3]
    n_cal2 = int((code == splits.SPLIT_CODES['cal2']).sum())
    need = [math.ceil(2000 / r.release_rate) if r.release_rate > 0 else N2_CAP]
    for y in (0, 1):
        rr = r[f'release_rate_class{y}']
        need.append(math.ceil(1000 / rr) if rr > 0 else N2_CAP)
    n2 = int(min(N2_CAP, n_cal2, max(need)))
    alpha_sel, projection = {}, {}
    for y in (0, 1):
        a_exp = int(round(r[f'release_rate_class{y}'] * n2))
        rate = r[f'selective_error_class{y}'] if a_exp > 0 and not np.isnan(r[f'selective_error_class{y}']) else 1.0
        u = float(cp_upper(int(round(rate * a_exp)), a_exp, DELTA2 / 2)) if a_exp > 0 else 1.0
        alpha_sel[y] = dev.smallest_at_or_above(u, ALPHA_SEL_GRID[y])
        projection[y] = {'expected_accepted': a_exp, 'development_selective_error': float(rate), 'projected_upper': u}
    TARGETS[task] = {'n_cal2_explained': n2, 'alpha_sel': alpha_sel, 'projection': projection}
    print(task, json.dumps(TARGETS[task], indent=1, default=float))

## Final model on the whole training portion

In [ ]:
FINAL = {}
for task in TASKS:
    Xt, yt, names, code, train_rows = DATA[task]
    ch = CHOICE[task]
    t0 = time.time()
    model, ens, pred = dev.fit_exported(Xt, yt, ch['params'], SEED, names)
    ens.meta.update({'task': task, 'config': ch['config'], 'params': ch['params'], 'seed': SEED,
                     'tau_rule': 'Youden on 5-fold out-of-fold scores of the training portion'})
    model_file = FROZEN_DIR / f'{task}__model.json'
    ens.save(model_file)
    nat = native_scores(model, 'xgboost', Xt)
    exp_pred = pred.predict(Xt)
    dis = np.flatnonzero(exp_pred != (nat > ens.tau).astype(np.int8))
    far = [int(i) for i in dis if abs(nat[i] - ens.tau) > 1e-5]
    g = pred.grid
    ib = IntervalBound(pred); cs = CellSearch(ib); rs = RealSearch(ens, g.thresholds)
    zr = Z3RealEncoding(ens, g.thresholds, timeout_ms=20000)
    rng = np.random.default_rng(SEED)
    pick = rng.choice(len(Xt), 300, replace=False)
    C = g.cells(Xt[pick]); yh = pred.predict_cells(C)
    checks = {'boxes': 300, 'primary_vs_second_disagree': 0, 'z3_checked': 0, 'z3_unknown': 0, 'z3_disagree': 0, 'sound': 0}
    for i in range(len(C)):
        box = list(g.point_box(C[i]))
        for j in rng.choice(g.active, size=int(rng.integers(3, min(18, len(g.active)) + 1)), replace=False):
            box[j] = (0, int(g.K[j]))
        box = tuple(box); yb = int(yh[i])
        a = cs.check(box, yb); b = rs.check(box, yb)
        checks['sound'] += a == 'sound'
        checks['primary_vs_second_disagree'] += ('unknown' not in (a, b)) and a != b
        if i < 30:
            c = zr.check(box, yb); checks['z3_checked'] += 1; checks['z3_unknown'] += c == 'unknown'
            checks['z3_disagree'] += (c != 'unknown') and (a != 'unknown') and c != a
    assert not far and checks['primary_vs_second_disagree'] == 0 and checks['z3_disagree'] == 0, 'frozen model fails G2'
    FINAL[task] = {'model': model, 'ens': ens, 'pred': pred, 'model_file': model_file, 'model_sha256': ens.hash(),
                   'exported_vs_native_disagreements': len(dis), 'soundness_checks': checks, 'thresholds': int(g.K.sum()),
                   'log10_family_size': g.family_size_log10(), 'seconds': time.time() - t0}
    print(task, {k: v for k, v in FINAL[task].items() if k not in ('model', 'ens', 'pred')})

## Frozen prior from 2,000 training patients

In [ ]:
for task in TASKS:
    Xt, yt, names, code, train_rows = DATA[task]
    pred = FINAL[task]['pred']; g = pred.grid; lam = CHOICE[task]['lambda']
    gen = CandidateGenerator(pred, M=M, seed=SEED)
    cache_path = FROZEN_DIR / f'{task}__prior_cache.pkl'
    gen.load(cache_path)
    rng = np.random.default_rng(SEED + 2)
    pr = np.sort(rng.choice(len(Xt), N_PRIOR_FINAL, replace=False))
    C = g.cells(Xt[pr]); yh = pred.predict_cells(C)
    t0 = time.time()
    lists = dev.candidate_sets(gen, C, yh, log=print, every=500)
    gen.save(cache_path)
    boxes = sorted({(c.key, c.y) for v in lists for c in v})
    verdicts, _ = verify_boxes(g, boxes)
    sub = {gen.cell_key(C[i]): gen.cache[gen.cell_key(C[i])] for i in range(len(C))}
    removed = drop_unverified(sub, verdicts)
    lists = [sub[gen.cell_key(C[i])] for i in range(len(C))]
    z3res, _ = z3_sample(g, [b for b in boxes if verdicts[b] == 'sound'], 30, SEED, timeout_ms=10000, time_budget_s=120)
    counts = dev.prior_counts(lists, M)
    prior = {'type': 'mixture', 'lambda': lam, 'd': int(g.d), 'K': g.K.tolist(), 'M': M, 'training_patients': N_PRIOR_FINAL,
             'training_rows': [int(train_rows[i]) for i in pr], 'counts': [[[list(t) for t in k], v] for k, v in sorted(counts.items())]}
    prior_file = FROZEN_DIR / f'{task}__prior.json'
    json.dump(prior, open(prior_file, 'w'))
    FINAL[task].update({'prior_file': prior_file, 'prior_sha256': sha256_file(prior_file), 'prior_boxes': len(counts),
                        'prior_unverified_removed': int(removed), 'prior_z3_sample': z3res,
                        'prior_seconds': time.time() - t0,
                        'prior_median_seconds_per_patient': float(np.median([gen.seconds[gen.cell_key(c)] for c in C]))})
    print(task, f"prior: {len(counts)} boxes, removed {removed}, Z3 sample {z3res}, {time.time() - t0:.0f}s")

## Explained subsamples (row indices only) and the frozen configuration

In [ ]:
FROZEN_SHA = {}
for task in TASKS:
    Xt, yt, names, code, train_rows = DATA[task]
    rng = np.random.default_rng(SEED + 1)
    test_rows = splits.indices(code, 'test'); cal2_rows = splits.indices(code, 'cal2')
    test_explained = np.sort(rng.choice(test_rows, N_TEST_EXPLAIN, replace=False))
    cal2_explained = np.sort(rng.choice(cal2_rows, TARGETS[task]['n_cal2_explained'], replace=False))
    np.save(FROZEN_DIR / f'{task}__test_explained.npy', test_explained)
    np.save(FROZEN_DIR / f'{task}__cal2_explained.npy', cal2_explained)
    ch, fin = CHOICE[task], FINAL[task]
    mr = DEV_ALL[(DEV_ALL.task == task) & (DEV_ALL.config == ch['config']) & (DEV_ALL['lambda'] == ch['lambda'])
                 & (DEV_ALL.M == M) & (DEV_ALL.alpha_0 == ALPHA0_MAIN) & (DEV_ALL.beta == BETA_MAIN)].iloc[0]
    data_file = public.SPECS[task].file
    cfg = {
        'task': task, 'outcome': public.SPECS[task].outcome, 'outcome_definition': public.SPECS[task].outcome_definition,
        'data_sha256': CARDS['sha256'][task], 'split_sha256': splits.summary(code)['sha256'], 'seed': SEED,
        'model': {'family': 'xgboost', 'config': ch['config'], 'params': ch['params'], 'sha256': fin['model_sha256'],
                  'tau': fin['ens'].tau, 'tau_rule': 'Youden on 5-fold out-of-fold scores of the training portion',
                  'file': f"datasets/derived/frozen/{task}__model.json", 'trained_on': 'training portion (dev-train and dev-cal)'},
        'candidates': {'M': M, 'order_seed': SEED, 'node_budget': 200000,
                       'inflation': 'binary search per side, every sound visited box kept',
                       'soundness': 'interval pre-check, then exact leaf branch-and-bound; every candidate re-checked by the second exact implementation'},
        'prior': {'type': 'mixture', 'lambda': ch['lambda'], 'training_patients': N_PRIOR_FINAL,
                  'file': f"datasets/derived/frozen/{task}__prior.json", 'sha256': fin['prior_sha256']},
        'targets': {'delta_1': DELTA1, 'delta_2': DELTA2, 'alpha_0': ALPHA0_MAIN, 'alpha_0_grid': ALPHA0_GRID,
                    'beta': BETA_MAIN, 'beta_grid': BETA_GRID, 'alpha_1': round(1 - BETA_MAIN, 10),
                    'alpha_sel': {'negative_outcome': TARGETS[task]['alpha_sel'][0], 'enrichment': TARGETS[task]['alpha_sel'][1]},
                    'policy': 'negative-outcome and enrichment releases certified separately at delta_2 / 2 each'},
        'release_rule': 'fewest constrained features, then most calibration support, then tightest bound; abstain if no candidate passes',
        'explained': {'test': N_TEST_EXPLAIN, 'test_sha256': sha256_array(test_explained),
                      'cal2': int(len(cal2_explained)), 'cal2_sha256': sha256_array(cal2_explained)},
        'development': {'g1_reference_auroc': G1_REF[task], 'target_negative_release': TARGET_NEGATIVE_RELEASE,
                        'target_met': ch['target_met'], 'release_rate_negative': float(mr.release_rate_class0),
                        'release_rate_enrichment': float(mr.release_rate_class1),
                        'selective_error_negative': float(mr.selective_error_class0),
                        'selective_error_enrichment': float(mr.selective_error_class1),
                        'alpha_sel_projection': TARGETS[task]['projection']},
        'frozen_model_checks': {'exported_vs_native_disagreements': fin['exported_vs_native_disagreements'],
                                'soundness_checks': fin['soundness_checks'], 'thresholds': fin['thresholds'],
                                'log10_family_size': fin['log10_family_size'], 'prior_boxes': fin['prior_boxes'],
                                'prior_unverified_removed': fin['prior_unverified_removed'], 'prior_z3_sample': fin['prior_z3_sample']},
        'calibration_split_1_opened': False,
    }
    FROZEN_SHA[task] = write_frozen(f'{REPO_DIR}/configs/frozen/{task}.json', cfg)
    print(task, FROZEN_SHA[task])
    print(json.dumps({k: cfg[k] for k in ('model', 'prior', 'targets', 'explained', 'development')}, indent=1, default=float))
write_json(f'{paths.RESULTS}/05_frozen.json', {'frozen_config_sha256': FROZEN_SHA, 'written_before_calibration_split_1': True,
                                               'choice': CHOICE, 'targets': TARGETS})

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "05: development and freezing for tasks B and A (gates G1 and G3, policy targets, final models, priors, explained subsamples, frozen configurations)"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)